# 02 — Neural Networks, Loss, Gradients, and Training

**Network LLM Engineering — Part I — Foundations**

### Learning goals
- Understand parameters, forward pass, loss, backpropagation, optimizer
- See why learning rate and batches matter
- Train a tiny classifier on networking features

## From networking intuition to ML intuition

Think of a model as a parameterized function. During a **forward pass** it maps inputs to predictions.
A **loss function** measures error. **Backpropagation** computes how each parameter contributed to that error.
An **optimizer** changes parameters to reduce future loss.

This is conceptually similar to a control loop: measure error -> compute correction -> update -> measure again.
The analogy is imperfect, but useful.

In [ ]:
import torch
torch.manual_seed(42)

# Tiny synthetic incident classifier:
# x = [packet_loss_pct, crc_errors, dns_failures]
X = torch.tensor([
    [0.0, 0.0, 1.0],
    [0.0, 0.0, 1.0],
    [4.0, 0.0, 0.0],
    [8.0, 0.0, 0.0],
    [0.0, 120.0, 0.0],
    [0.0, 300.0, 0.0],
], dtype=torch.float32)
y = torch.tensor([0,0,1,1,2,2])
model = torch.nn.Sequential(torch.nn.Linear(3, 8),torch.nn.ReLU(),torch.nn.Linear(8, 3))
opt = torch.optim.AdamW(model.parameters(), lr=0.03)
loss_fn = torch.nn.CrossEntropyLoss()
for epoch in range(120):
    logits = model(X)
    loss = loss_fn(logits, y)
    opt.zero_grad(); loss.backward(); opt.step()
print("loss:", float(loss))
print("pred:", model(X).argmax(-1).tolist())
print("true:", y.tolist())

### Important terms

- **Logit:** raw model score before softmax.
- **Softmax:** turns logits into probabilities.
- **Cross-entropy:** penalizes low probability on the correct class/token.
- **Gradient:** local direction of loss change.
- **Learning rate:** how aggressively the optimizer updates weights.
- **Epoch:** one pass over the dataset.
- **Batch:** examples processed together.

### Exercise

Change the learning rate to `0.00001` and then `1.0`.
Observe that "bigger learning rate = faster learning" is not generally true.